In [94]:
import pandas as pd
import numpy as np


# ============================================================
# 1. LOAD DATA
# ============================================================

def load_data(url):
    df = pd.read_csv(url)

    columns = [
        'engine_displacement',
        'horsepower',
        'vehicle_weight',
        'model_year',
        'fuel_efficiency_mpg'
    ]

    return df[columns]


# ============================================================
# 2. PREPARE X
# ============================================================

def prepare_X(df, columns, fill_value):
    """
    Prepare the feature matrix X.

    - Select features
    - Fill missing values
    - Convert to numpy array
    - Add a column of ones for the intercept
    """

    df = df.copy()

    df[columns] = df[columns].fillna(fill_value)

    X = df[columns].values

    # Add bias/intercept column
    X = np.column_stack([np.ones(len(X)), X])

    return X


# ============================================================
# 3. PREPARE y
# ============================================================

def prepare_y(df, target):
    return df[target].values


# ============================================================
# 4. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

def split_data(df, seed=42):

    n = len(df)

    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)

    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    return df_train, df_val, df_test


# ============================================================
# 5. LINEAR REGRESSION FROM SCRATCH
# ============================================================

def linear_regression(X, y, r=0):
    """
    Linear regression using the normal equation.

    r = 0       -> ordinary linear regression
    r > 0       -> regularized linear regression

    The intercept is NOT regularized.
    """

    XTX = X.T @ X
    XTy = X.T @ y

    # Regularization matrix
    reg = np.eye(X.shape[1]) * r

    # Do not regularize the intercept
    reg[0, 0] = 0

    w = np.linalg.solve(
        XTX + reg,
        XTy
    )

    return w


# ============================================================
# 6. PREDICT
# ============================================================

def predict(X, w):
    return X @ w


# ============================================================
# 7. RMSE
# ============================================================

def rmse(y, y_pred):

    error = y_pred - y

    mse = np.mean(error ** 2)

    return np.sqrt(mse)


# ============================================================
# 8. TRAIN + VALIDATE
# ============================================================

def train_and_evaluate(
    df_train,
    df_val,
    features,
    target,
    fill_value=0,
    r=0
):

    X_train = prepare_X(
        df_train,
        features,
        fill_value
    )

    y_train = prepare_y(
        df_train,
        target
    )

    X_val = prepare_X(
        df_val,
        features,
        fill_value
    )

    y_val = prepare_y(
        df_val,
        target
    )

    # Train
    w = linear_regression(
        X_train,
        y_train,
        r
    )

    # Predict
    y_pred = predict(
        X_val,
        w
    )

    # RMSE
    score = rmse(
        y_val,
        y_pred
    )

    return score


# ============================================================
# 9. COMPARE RMSE: ZERO vs MEDIAN
# ============================================================

def compare_imputation(
    df_train,
    df_val,
    features,
    target
):

    # Fill with 0
    rmse_zero = train_and_evaluate(
        df_train,
        df_val,
        features,
        target,
        fill_value=0,
        r=0
    )

    # Calculate median from TRAINING DATA ONLY
    median = df_train['horsepower'].median()

    # Fill with training median
    rmse_median = train_and_evaluate(
        df_train,
        df_val,
        features,
        target,
        fill_value=median,
        r=0
    )

    print("RMSE with 0:     ", round(rmse_zero, 3))
    print("RMSE with median:", round(rmse_median, 3))

    if rmse_zero < rmse_median:
        print("Better option: With 0")
    elif rmse_median < rmse_zero:
        print("Better option: With median")
    else:
        print("Both are equally good")

    return rmse_zero, rmse_median


# ============================================================
# 10. TEST REGULARIZATION FACTORS
# ============================================================

def test_regularization(
    df_train,
    df_val,
    features,
    target,
    r_values
):

    results = []

    for r in r_values:

        score = train_and_evaluate(
            df_train,
            df_val,
            features,
            target,
            fill_value=0,
            r=r
        )

        results.append(
            (r, score)
        )

    print("Regularization results:")
    print()

    for r, score in results:
        print(
            f"r = {r:<6} "
            f"RMSE = {score:.4f}"
        )

    # Find lowest RMSE
    best_r, best_score = min(
        results,
        key=lambda x: x[1]
    )

    print()
    print("Best r:", best_r)
    print("Best RMSE:", round(best_score, 4))

    return results


# ============================================================
# 11. TEST EFFECT OF RANDOM SEED
# ============================================================

def test_seeds(
    df,
    features,
    target,
    seeds
):

    scores = []

    for seed in seeds:

        # Create split using this seed
        df_train, df_val, df_test = split_data(
            df,
            seed
        )

        # Train without regularization
        score = train_and_evaluate(
            df_train,
            df_val,
            features,
            target,
            fill_value=0,
            r=0
        )

        scores.append(score)

        print(
            f"Seed {seed}: "
            f"RMSE = {score:.6f}"
        )

    # Standard deviation
    std = np.std(scores)

    print()
    print("Standard deviation:", std)
    print("Rounded:", round(std, 3))

    return scores, std


# ============================================================
# 12. FINAL TEST WITH SEED 9
# ============================================================

def final_test(
    df,
    features,
    target,
    seed=9,
    r=0.001
):

    # Split
    df_train, df_val, df_test = split_data(
        df,
        seed
    )

    # Combine train + validation
    df_train_full = pd.concat(
        [df_train, df_val],
        ignore_index=True
    )

    # Prepare training data
    X_train = prepare_X(
        df_train_full,
        features,
        fill_value=0
    )

    y_train = prepare_y(
        df_train_full,
        target
    )

    # Prepare test data
    X_test = prepare_X(
        df_test,
        features,
        fill_value=0
    )

    y_test = prepare_y(
        df_test,
        target
    )

    # Train
    w = linear_regression(
        X_train,
        y_train,
        r
    )

    # Predict
    y_pred = predict(
        X_test,
        w
    )

    # RMSE
    score = rmse(
        y_test,
        y_pred
    )

    print("Final test RMSE:", score)
    print("Rounded:", round(score, 3))

    return score

In [95]:
print(df.columns[df.isnull().any()])

Index(['horsepower'], dtype='str')


In [96]:
print(df['horsepower'].median())

254.0


In [97]:
df_train, df_val, df_test = split_data(
    df,
    seed=42
)

In [98]:
rmse_zero, rmse_median = compare_imputation(
    df_train,
    df_val,
    features,
    target
)

RMSE with 0:      2.205
RMSE with median: 2.202
Better option: With median


In [99]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]

results = test_regularization(
    df_train,
    df_val,
    features,
    target,
    r_values
)

Regularization results:

r = 0      RMSE = 2.2053
r = 0.01   RMSE = 2.2053
r = 0.1    RMSE = 2.2053
r = 1      RMSE = 2.2053
r = 5      RMSE = 2.2053
r = 10     RMSE = 2.2053
r = 100    RMSE = 2.2053

Best r: 0
Best RMSE: 2.2053


In [100]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

scores, std = test_seeds(
    df,
    features,
    target,
    seeds
)

Seed 0: RMSE = 2.239204
Seed 1: RMSE = 2.202113
Seed 2: RMSE = 2.163420
Seed 3: RMSE = 2.204359
Seed 4: RMSE = 2.201880
Seed 5: RMSE = 2.245785
Seed 6: RMSE = 2.269721
Seed 7: RMSE = 2.190795
Seed 8: RMSE = 2.216611
Seed 9: RMSE = 2.207037

Standard deviation: 0.0287812740782067
Rounded: 0.029


In [101]:
test_score = final_test(
    df,
    features,
    target,
    seed=9,
    r=0.001
)

Final test RMSE: 2.2357747028137993
Rounded: 2.236
